# Agrupamentos e Análise Comparativa

Agrupamentos avançados, tabelas dinâmicas e comparações com indicadores e exercícios comentados.

Os exemplos usam arquivos pequenos da pasta `datasets/` e podem ser executados na ordem. Exercícios resolvidos aparecem junto dos respectivos enunciados.

**Objetivo:** comparar grupos de maneira consistente, distinguindo contagem, média, taxa e participação. Uma linha representa uma observação de venda sintética; duplicatas exatas são removidas neste exercício como hipótese explícita, não como regra universal.

### Materiais complementares

- [Guia Técnico](./reference_materials/03_agrupamentos_e_analise_comparativa.pdf)
- [Cheat Sheet](./cheat_sheets/cheat_sheet_agrupamentos.pdf)

Os arquivos estão nesta mesma pasta da disciplina. Os PDFs complementam, mas não substituem, a execução das células.

## 1. Agrupamentos simples e múltiplos

O notebook introdutório de Pandas usa `groupby()` para somar unidades por produto. Aqui ampliamos para múltiplos indicadores, categorias e interpretação. Os dados são sintéticos.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

def preparar_pastas():
    """Localiza a biblioteca e cria as pastas de dados e resultados, se ausentes."""
    atual = Path.cwd().resolve()
    # Aceita execução na raiz da biblioteca ou em uma pasta filha.
    candidatos = [atual, atual.parent]
    raiz = next(
        (p for p in candidatos if (p / 'vendas_exemplo.csv').is_file() or
         (p / 'datasets' / 'vendas_exemplo.csv').is_file()),
        None,
    )
    if raiz is None:
        # Identifica a raiz da biblioteca pelo conjunto de notebooks.
        raiz = next(
            (p for p in candidatos if (p / '01_Analise_Exploratoria_de_Dados.ipynb').is_file()),
            atual,
        )
    dados = raiz / 'datasets'
    saidas = raiz / 'outputs'
    dados.mkdir(parents=True, exist_ok=True)
    saidas.mkdir(parents=True, exist_ok=True)
    return dados, saidas

DATA, OUTPUTS = preparar_pastas()

def conferir_datasets(*nomes):
    ausentes = [nome for nome in nomes if not (DATA / nome).is_file()]
    if ausentes:
        raise FileNotFoundError(
            'Datasets ausentes: ' + ', '.join(ausentes) + '\n'
            + 'Coloque os CSVs fornecidos em: ' + str(DATA.resolve()) + '\n'
            + 'Criar a pasta datasets/ não cria automaticamente os arquivos CSV.'
        )
conferir_datasets('vendas_exemplo.csv', 'reclamacoes_sinteticas.csv', 'regioes_exemplo.csv')
vendas=pd.read_csv(DATA/'vendas_exemplo.csv',parse_dates=['data']).drop_duplicates()
vendas['receita']=vendas['unidades']*vendas['preco_unitario']
resumo=vendas.groupby('produto',as_index=False).agg(unidades_total=('unidades','sum'),receita_conhecida=('receita', lambda s: s.sum(min_count=1)),preco_medio=('preco_unitario','mean'),registros=('produto','size'))
print(resumo.round(2).to_string(index=False))

produto  unidades_total  receita_conhecida  preco_medio  registros
   Café             412             7416.0         18.0         12
    Chá             551             7714.0         14.0         12
   Suco             644             7188.0         12.0         12
   Água             768             5376.0          7.0         12


## 2. Comparações por duas dimensões

Cuidado com comparações de totais entre grupos com quantidades de observações diferentes. Considere médias, taxas ou proporções, conforme a pergunta.

In [2]:
comparacao=vendas.groupby(['regiao','canal'],dropna=False).agg(unidades=('unidades','sum'),receita=('receita', lambda s: s.sum(min_count=1))).reset_index()
print(comparacao.to_string(index=False))

  regiao  canal  unidades  receita
Nordeste Online       514   6243.0
   Norte   Loja       599   6976.0
 Sudeste   Loja       610   6589.0
     Sul Online       602   6986.0
     NaN Online        50    900.0


## 3. Pivot tables, crosstab e transform

`pivot_table` resume valores, `crosstab` conta frequências e `transform` devolve um resultado alinhado linha a linha, permitindo calcular participações.

In [3]:
pivot=vendas.pivot_table(index='produto',columns='canal',values='unidades',aggfunc='sum',fill_value=0)
print(pivot)
print(pd.crosstab(vendas['regiao'],vendas['canal'],dropna=False))
vendas['total_produto']=vendas.groupby('produto')['unidades'].transform('sum')
vendas['participacao_no_produto']=vendas['unidades']/vendas['total_produto']
assert np.allclose(vendas.groupby('produto')['participacao_no_produto'].sum(),1)

canal    Loja  Online
produto              
Café      175     237
Chá       306     245
Suco      315     329
Água      413     355
canal     Loja  Online
regiao                
Nordeste     0      11
Norte       12       0
Sudeste     12       0
Sul          0      12
NaN          0       1


## 4. Estudo de caso: reclamações por categoria

Este dataset é sintético e não representa empresas ou reclamações reais. Compare volume, tempo de resposta e taxa de resolução.

In [4]:
reclamacoes=pd.read_csv(DATA/'reclamacoes_sinteticas.csv',parse_dates=['data'])
resultado=reclamacoes.groupby('categoria').agg(quantidade=('id','size'),tempo_medio_dias=('tempo_resposta_dias','mean'),taxa_resolucao=('resolvida','mean'))
resultado['taxa_resolucao_pct']=(resultado['taxa_resolucao']*100).round(1)
print(resultado[['quantidade','tempo_medio_dias','taxa_resolucao_pct']].round(1))

              quantidade  tempo_medio_dias  taxa_resolucao_pct
categoria                                                     
Atendimento           41               7.6                68.3
Cancelamento          32               8.2                84.4
Cobrança              46               7.4                71.7
Entrega               41               7.6                70.7


## 5. Exercício resolvido — identificar categoria crítica

**Enunciado:** encontre a categoria com maior prazo médio de resposta, considerando todas as reclamações.

In [5]:
atraso=resultado['tempo_medio_dias'].idxmax()
print('Categoria com maior tempo médio de resposta:',atraso)
print('Dias:',round(resultado.loc[atraso,'tempo_medio_dias'],2))

Categoria com maior tempo médio de resposta: Cancelamento
Dias: 8.25


## 6. Exercício resolvido — verificar ponderações

**Enunciado:** compare a média geral de tempo de resposta com a média simples das médias de categoria. Explique por que podem ser diferentes.

In [6]:
media_geral=reclamacoes['tempo_resposta_dias'].mean()
media_das_medias=resultado['tempo_medio_dias'].mean()
print('Média geral:',round(media_geral,2),'| média das médias:',round(media_das_medias,2))
print('As categorias podem ter quantidades diferentes de registros; a média geral pondera pelo volume.')

Média geral: 7.68 | média das médias: 7.71
As categorias podem ter quantidades diferentes de registros; a média geral pondera pelo volume.


## 7. Agregações nomeadas, percentuais e comparação justa
Totais maiores podem refletir apenas mais registros. Compare volume, taxa por registro e participação percentual, informando o denominador.

In [7]:
base = vendas.copy()
base['receita'] = base['unidades'] * base['preco_unitario']
base['receita_conhecida'] = base['receita'].notna()
comparativo = (base.groupby('regiao', dropna=False)
               .agg(registros=('produto','size'), unidades=('unidades','sum'), receita=('receita', lambda s: s.sum(min_count=1)), registros_sem_preco=('preco_unitario', lambda s: s.isna().sum())))
comparativo['unidades_por_registro'] = comparativo['unidades'] / comparativo['registros']
comparativo['participacao_receita_pct'] = 100 * comparativo['receita'] / comparativo['receita'].sum(min_count=1)
print('A participação considera apenas receita conhecida; sem preços completos não representa a receita real total.')
print(comparativo.round(2).to_string())
assert np.isclose(comparativo['participacao_receita_pct'].sum(), 100)

A participação considera apenas receita conhecida; sem preços completos não representa a receita real total.
          registros  unidades  receita  registros_sem_preco  unidades_por_registro  participacao_receita_pct
regiao                                                                                                      
Nordeste         11       514   6243.0                    0                  46.73                     22.54
Norte            12       599   6976.0                    0                  49.92                     25.19
Sudeste          12       610   6589.0                    1                  50.83                     23.79
Sul              12       602   6986.0                    0                  50.17                     25.23
NaN               1        50    900.0                    0                  50.00                      3.25


## 8. `merge`, `join` e `concat`: objetivos diferentes
`merge` associa informações por chaves; `join` normalmente alinha por índice; `concat` empilha ou justapõe tabelas. Em relações muitos-para-um, `validate='many_to_one'` evita multiplicação inesperada de linhas.

In [8]:
referencia = pd.read_csv(DATA/'regioes_exemplo.csv')
ligada = base.merge(referencia, on='regiao', how='left', validate='many_to_one', indicator=True)
print(ligada['_merge'].value_counts().to_string())
assert len(ligada) == len(base)
primeira, segunda = base.iloc[:10].copy(), base.iloc[10:].copy()
recomposta = pd.concat([primeira, segunda], ignore_index=True)
assert len(recomposta) == len(base)

_merge
both          47
left_only      1
right_only     0


## 9. Exercício resolvido — ranking por categoria e canal
**Enunciado:** calcule receita por produto e canal; para cada canal, determine o produto líder em receita. Confirme que o total do agrupamento coincide com o total da base.

In [9]:
ranking = (base.groupby(['canal','produto'], dropna=False, as_index=False)
           .agg(receita=('receita', lambda s: s.sum(min_count=1)), unidades=('unidades','sum')))
indices = ranking.groupby('canal')['receita'].idxmax()
lideres = ranking.loc[indices].sort_values('canal')
print(lideres.to_string(index=False))
assert np.isclose(ranking['receita'].sum(), base['receita'].sum())

 canal produto  receita  unidades
  Loja     Chá   4284.0       306
Online    Café   4266.0       237


## 10. Exercício resolvido — taxa de resolução e risco de médias ingênuas
**Enunciado:** compare categorias por taxa de resolução e tempo médio de resposta, observando o tamanho de cada grupo. A média simples das médias pode distorcer a média global.

In [10]:
rec = pd.read_csv(DATA/'reclamacoes_sinteticas.csv')
por_categoria = (rec.groupby('categoria').agg(
   n=('id', 'size'), taxa_resolucao=('resolvida','mean'), tempo_medio=('tempo_resposta_dias','mean')))
por_categoria['taxa_resolucao_pct'] = (100*por_categoria['taxa_resolucao']).round(1)
print(por_categoria[['n', 'taxa_resolucao_pct', 'tempo_medio']].round(2).to_string())
media_ponderada = np.average(por_categoria['tempo_medio'], weights=por_categoria['n'])
assert np.isclose(media_ponderada, rec['tempo_resposta_dias'].mean())

               n  taxa_resolucao_pct  tempo_medio
categoria                                        
Atendimento   41                68.3         7.61
Cancelamento  32                84.4         8.25
Cobrança      46                71.7         7.43
Entrega       41                70.7         7.56


### Consulta rápida — agregação
- `size()` conta linhas, inclusive linhas com valores ausentes em outras colunas.
- `count()` conta somente valores **não nulos** da coluna escolhida.
- `transform()` mantém a quantidade de linhas da tabela original.
- `pivot_table()` agrega valores; `crosstab()` conta combinações de categorias.
- Confira sempre se somas antes e depois do agrupamento coincidem.

## 11. Exercício resolvido — taxas, denominadores e dados ausentes

**Enunciado:** calcule a taxa de resolução por categoria por duas vias (média de indicador binário e razão entre casos resolvidos e casos válidos). Confira os denominadores e identifique grupos com preços ausentes nas vendas.

**Por que isso importa?** Uma taxa é uma razão entre eventos e oportunidades. `size()` inclui registros mesmo quando o indicador está ausente; `count()` não. A receita desconhecida não equivale a zero.

In [11]:
taxas = reclamacoes.groupby('categoria').agg(
    registros=('id', 'size'),
    respostas_validas=('resolvida', 'count'),
    resolvidas=('resolvida', 'sum'),
    taxa_media=('resolvida', 'mean')
)
taxas['taxa_por_validos'] = taxas['resolvidas'] / taxas['respostas_validas']
assert np.allclose(taxas['taxa_media'], taxas['taxa_por_validos'], equal_nan=True)
print(taxas.round(3).to_string())
qualidade_receita = vendas.groupby('produto').agg(
    vendas=('produto', 'size'), precos_ausentes=('preco_unitario', lambda s: s.isna().sum()),
    receita_conhecida=('receita', lambda s: s.sum(min_count=1))
)
print('\nCobertura de preços por produto:\n', qualidade_receita)
assert qualidade_receita['precos_ausentes'].sum() == vendas['preco_unitario'].isna().sum()

              registros  respostas_validas  resolvidas  taxa_media  taxa_por_validos
categoria                                                                           
Atendimento          41                 41          28       0.683             0.683
Cancelamento         32                 32          27       0.844             0.844
Cobrança             46                 46          33       0.717             0.717
Entrega              41                 41          29       0.707             0.707

Cobertura de preços por produto:
          vendas  precos_ausentes  receita_conhecida
produto                                            
Café         12                0             7416.0
Chá          12                0             7714.0
Suco         12                1             7188.0
Água         12                0             5376.0


### Interpretação e boas práticas

- A média de um indicador 0/1 corresponde à proporção de `1` **entre registros válidos**; se houver valores nulos, o denominador exige atenção.
- Médias por categoria não devem ser promediadas sem ponderação quando os grupos têm tamanhos distintos.
- `sum(min_count=1)` diferencia ausência total de valores de soma igual a zero.
- Rankings feitos somente com receita conhecida são descrições parciais, não totais garantidos.

## Consulta rápida

`groupby().agg()` agrega várias métricas; `transform()` atribui um resultado por linha; `pivot_table()` resume valores; `crosstab()` conta frequências. Para grupos vazios, dados ausentes e denominadores pequenos, explicite o critério adotado.

| Situação | Recomendação |
|---|---|
| Nulos na medida agregada | `sum(min_count=1)` e contagem de válidos |
| Médias de grupos desiguais | ponderar pelo número de casos válidos |
| Comparação de taxas | declarar numerador e denominador |
| Chaves de associação | `merge(..., validate=...)` |

## Aprofundamento v2 — média ponderada e efeito do denominador

Comparar médias de grupos de tamanhos diferentes sem declarar pesos pode produzir conclusões erradas. Em especial, a média simples das médias de grupos **não** equivale à média global quando o número de observações difere.

**Exercício resolvido:** comparar agregação ponderada e não ponderada em uma amostra explícita.

In [12]:
import numpy as np
import pandas as pd
exemplo_pesos_v2 = pd.DataFrame({'grupo':['A','A','A','B'], 'nota':[5.,5.,5.,1.]})
resumo_pesos_v2 = exemplo_pesos_v2.groupby('grupo')['nota'].agg(['mean','size'])
media_grupos_v2 = resumo_pesos_v2['mean'].mean()
media_ponderada_v2 = np.average(resumo_pesos_v2['mean'],weights=resumo_pesos_v2['size'])
assert np.isclose(media_ponderada_v2, exemplo_pesos_v2['nota'].mean())
assert not np.isclose(media_grupos_v2,media_ponderada_v2)
print('Média não ponderada dos grupos:', media_grupos_v2)
print('Média ponderada por observações:',media_ponderada_v2)

Média não ponderada dos grupos: 3.0
Média ponderada por observações: 4.0


**Interpretação:** a ponderação depende da pergunta de pesquisa. Usar grupos com peso igual pode ser legítimo, desde que essa escolha seja explicitada.